In [1]:
from pathlib import Path
from tqdm import tqdm
import SimpleITK as sitk
import numpy as np
import pandas as pd

OUT_DIR = Path("/home/user/nschuler/fullbody-sCT/preprocessing/preprocessing_synthrad/analysis")
BASE_ROOT = Path("/local/scratch/datasets/FullbodySCT/Synthrad_combined_preprocessed")
src_root = BASE_ROOT / "1initNifti"

CT_BACKGROUND = -1024
MR_BACKGROUND = 0
MASK_BACKGROUND = 0

In [2]:
def get_first_nifti(folder: Path):
    if not folder.exists():
        return None
    for pattern in ("*.nii.gz", "*.nii"):
        files = sorted(folder.glob(pattern))
        if files:
            return files[0]
    return None


In [3]:
def collect_metadata(src_root: Path) -> pd.DataFrame:
    rows = []

    for case_dir in tqdm(sorted(src_root.iterdir())):
        if not case_dir.is_dir():
            continue

        case_id = case_dir.name

        ct_in = get_first_nifti(case_dir / "CT_reg")
        mr_in = get_first_nifti(case_dir / "MR")
        mask_in = get_first_nifti(case_dir / "masks")

        def process_one(modality: str, in_path: Path):
            img = sitk.ReadImage(str(in_path))
            arr = sitk.GetArrayFromImage(img)  # (z, y, x)
            z, y, x = arr.shape

            sx, sy, sz = img.GetSpacing()  # (sx, sy, sz) in physical coords

            rows.append({
                "case_id": case_id,
                "modality": modality,
                "filename": in_path.name,
                "z": z,
                "y": y,
                "x": x,
                "spacing_x": sx,
                "spacing_y": sy,
                "spacing_z": sz,
            })

        if ct_in is not None:
            process_one("CT", ct_in)
        if mr_in is not None:
            process_one("MR", mr_in)
        if mask_in is not None:
            process_one("MASK", mask_in)

    return pd.DataFrame(rows)


In [4]:
# df = collect_metadata(src_root)
# df.head()


In [5]:
# df.to_csv(OUT_DIR / "dataset_shape_spacing_stats.csv", index=False)

In [6]:
df = pd.read_csv(OUT_DIR / "dataset_shape_spacing_stats.csv")
df.head()

df.sample(20)

,case_id,modality,filename,z,y,x,spacing_x,spacing_y,spacing_z
2810,pelvis_1PC097,MASK,pelvis_1PC097_mask.nii.gz,84,312,417,1.0,1.0,2.5
288,AB_1ABB057,CT,AB_1ABB057_CT_reg.nii.gz,110,438,437,1.0,1.0,3.0
1555,TH_1THB054,MR,TH_1THB054_MR.nii.gz,142,527,592,1.0,1.0,3.0
1585,TH_1THB073,MR,TH_1THB073_MR.nii.gz,115,500,545,1.0,1.0,3.0
301,AB_1ABB062,MR,AB_1ABB062_MR.nii.gz,126,410,404,1.0,1.0,3.0
2700,pelvis_1PC038,CT,pelvis_1PC038_CT_reg.nii.gz,92,325,441,1.0,1.0,2.5
1157,HN_1HND092,MASK,HN_1HND092_mask.nii.gz,100,319,280,1.0,1.0,3.0
2078,brain_1BB182,MASK,brain_1BB182_mask.nii.gz,184,233,222,1.0,1.0,1.0
2698,pelvis_1PC037,MR,pelvis_1PC037_MR.nii.gz,87,289,418,1.0,1.0,2.5
1993,brain_1BB066,MR,brain_1BB066_MR.nii.gz,182,238,206,1.0,1.0,1.0


In [7]:
df = df[df["modality"]=="MR"]
df["body_part"] = df["case_id"].str.split("_").str[0]

desc = df.groupby("body_part")[["x", "y", "z"]].describe()

In [8]:
df.groupby("body_part")[["spacing_z", "spacing_y", "spacing_x"]].describe()

spacing_z                                    spacing_y       ...  \
              count mean  std  min  25%  50%  75%  max     count mean  ...   
body_part                                                              ...   
AB            175.0  3.0  0.0  3.0  3.0  3.0  3.0  3.0     175.0  1.0  ...   
HN            221.0  3.0  0.0  3.0  3.0  3.0  3.0  3.0     221.0  1.0  ...   
TH            182.0  3.0  0.0  3.0  3.0  3.0  3.0  3.0     182.0  1.0  ...   
brain         180.0  1.0  0.0  1.0  1.0  1.0  1.0  1.0     180.0  1.0  ...   
pelvis        180.0  2.5  0.0  2.5  2.5  2.5  2.5  2.5     180.0  1.0  ...   

                    spacing_x                                     
           75%  max     count mean  std  min  25%  50%  75%  max  
body_part                                                         
AB         1.0  1.0     175.0  1.0  0.0  1.0  1.0  1.0  1.0  1.0  
HN         1.0  1.0     221.0  1.0  0.0  1.0  1.0  1.0  1.0  1.0  
TH         1.0  1.0     182.0  1.0  0.0  1.0  1.0  1.0  1.0  1.0  
brain      1.0  1.0     180.0  1.0  0.0  1.0  1.0  1.0  1.0  1.0  
pelvis     1.0  1.0     180.0  1.0  0.0  1.0  1.0  1.0  1.0  1.0  

[5 rows x 24 columns]

In [9]:
def extract_hosp_char(case_id: str, body_part: str) -> str:
    s = case_id.upper()
    
    # mapping from body_part to the prefix we search in the case_id
    prefix_map = {
        "pelvis": "P",
        "brain": "B",
        "TH": "TH",
        "HN": "HN",
        "AB": "AB",
    }

    prefix = prefix_map.get(body_part, body_part)  # fallback: use body_part itself
    idx = s.rfind(prefix)

    pos = idx + len(prefix)
    if pos < len(s):
        hospital = s[pos]
        if body_part in ("pelvis", "brain"):
            return "2023" + hospital
        else:
            return "2025" + hospital
    else:
        return ""

df["hosp_char"] = df.apply(
    lambda row: extract_hosp_char(row["case_id"], row["body_part"]),
    axis=1,
)

In [10]:
df.groupby(["hosp_char", "body_part"])["spacing_x"].describe()

count  mean  std  min  25%  50%  75%  max
hosp_char body_part                                           
20230     brain       42.0   1.0  0.0  1.0  1.0  1.0  1.0  1.0
20231     brain       16.0   1.0  0.0  1.0  1.0  1.0  1.0  1.0
20232     brain        2.0   1.0  0.0  1.0  1.0  1.0  1.0  1.0
2023A     brain       60.0   1.0  0.0  1.0  1.0  1.0  1.0  1.0
          pelvis     120.0   1.0  0.0  1.0  1.0  1.0  1.0  1.0
2023C     brain       60.0   1.0  0.0  1.0  1.0  1.0  1.0  1.0
          pelvis      60.0   1.0  0.0  1.0  1.0  1.0  1.0  1.0
2025A     AB          65.0   1.0  0.0  1.0  1.0  1.0  1.0  1.0
          HN          91.0   1.0  0.0  1.0  1.0  1.0  1.0  1.0
          TH          91.0   1.0  0.0  1.0  1.0  1.0  1.0  1.0
2025B     AB          91.0   1.0  0.0  1.0  1.0  1.0  1.0  1.0
          TH          91.0   1.0  0.0  1.0  1.0  1.0  1.0  1.0
2025C     AB          19.0   1.0  0.0  1.0  1.0  1.0  1.0  1.0
          HN          65.0   1.0  0.0  1.0  1.0  1.0  1.0  1.0
2025D     HN          65.0   1.0  0.0  1.0  1.0  1.0  1.0  1.0